In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import pyspark


In [ ]:
spark = (SparkSession.builder.appName('dataframe_file')
         .config('spark.dynamicAllocation.enabled',"false")
         .config('spark.sql.adaptive.enabled',"false")
         .config("spark.executor.extraClassPath",'/home/daniel/jdbc_folder/postgresql42.7.0.jar')
         .config("spark.driver.extraClassPath", '/home/daniel/jdbc_folder/postgresql42.7.0.jar')
         .config("spark.jars.packages", "org.postgresql:postgresql:42.7.0") 
         .getOrCreate())   

In [ ]:
spark

In [ ]:
yellowtaxicsv = spark.read.csv('DataFiles/Raw/YellowTaxis_*.csv',header=True)

In [ ]:
yellowtaxicsv.show()

In [ ]:
yellowtaxicsv.printSchema()

In [ ]:
yellowtaxicsvde = yellowtaxicsv.describe(['passenger_count','trip_distance'])

In [ ]:
yellowtaxicsvde.show()

# performing data cleansing on dataset

In [ ]:

yellowTaxicleansing =( yellowtaxicsv.where('passenger_count > 0.0 ').where('trip_distance > 0.0').na.drop('all')
               .dropDuplicates().where('tpep_pickup_datetime > "2022-10-01" AND tpep_dropoff_datetime > "2022-11-01" '))


In [ ]:
yellowTaxicleansing.show()

In [ ]:
#total number of records after cleansing
yellowTaxicleansing.count()

## selecting the number of column to be used as per business logic

In [ ]:
yellowTaxicleansing.printSchema()

In [ ]:
selected_taxi_column = yellowTaxicleansing.select("VendorID",
                                                  col("passenger_count").cast('integer'),
                                                  col("store_and_fwd_flag").alias("flag"),
                                                 'PULocationID',
                                                 'DOLocationID',
                                                 'payment_type',
                                                 'fare_amount',
                                                 'extra',
                                                 'total_amount',
                                                'tip_amount',
                                                  col('tpep_dropoff_datetime').alias('dropoff_time'),
                                                  col('tpep_pickup_datetime').alias('pickup_time')  )

# renaming few columns

In [ ]:
selected_taxi_column = selected_taxi_column.withColumnRenamed('PULocationID','pickupLocationID').withColumnRenamed('DOLocationID','dropoffLocationID')

In [ ]:
selected_taxi_column.printSchema()

# creating an extra column (trip_year) ,extracting the year character from pickup_time

In [ ]:
selected_taxi_column = selected_taxi_column.withColumn('tripYear',year(col('pickup_time')))

selected_taxi_column.printSchema()

# grouping the data and performing some aggregation on it

In [ ]:
yellowTaxiReport = selected_taxi_column.groupBy('pickupLocationID','dropoffLocationID').agg(avg('tripYear').alias('avgTripTimes'),sum('total_amount')).alias('sumTotal').orderBy('pickupLocationID','dropoffLocationID')


yellowTaxiReport.show()

# performing optimization using persist() to cache data between disk and memory

In [ ]:
yellowTaxiReport.persist(pyspark.StorageLevel.MEMORY_AND_DISK)

# we would save to data to both disk and postgre database

In [ ]:
(yellowTaxiReport.write.option('header','true')
          .option('date_format','yyyy-MM-dd HH:mm:ss.S')
          .mode('overwrite').partitionBy('pickupLocationID')
          .csv('transformData'))

# writing to database

In [ ]:
db_properties = {}
dburl = database_url
db_properties['user'] =username
db_properties['password']= password


In [ ]:
yellowTaxiDb = yellowTaxiReport.write.jdbc(url= dburl,table='yellowtaxi',mode='overwrite',properties=db_properties)